In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [3]:
nav = pd.read_csv("C:\\Users\\tekam\\Downloads\\Data Science\\Data_Analyst_Portfolio_Project\\data\\raw\\nav_history_clean.csv")

fund = pd.read_csv("C:\\Users\\tekam\\Downloads\\Data Science\\Data_Analyst_Portfolio_Project\\data\\raw\\01_fund_master.csv")

nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(["amfi_code","date"])

display(nav.head())

display(fund.head())

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [4]:
df = nav.merge(

    fund[["amfi_code","scheme_name"]],

    on="amfi_code",

    how="left"

)

print(df.shape)

display(df.head())

(46000, 4)


,amfi_code,date,nav,scheme_name
0,100016,2022-01-03,520.4608,HDFC Top 100 Fund - Regular Plan - Growth
1,100016,2022-01-04,515.0971,HDFC Top 100 Fund - Regular Plan - Growth
2,100016,2022-01-05,521.7239,HDFC Top 100 Fund - Regular Plan - Growth
3,100016,2022-01-06,515.7880,HDFC Top 100 Fund - Regular Plan - Growth
4,100016,2022-01-07,515.1639,HDFC Top 100 Fund - Regular Plan - Growth


In [5]:
def calculate_cagr(data, years):
    
    result = []

    latest_date = data["date"].max()

    start_date = latest_date - pd.DateOffset(years=years)

    for scheme in data["scheme_name"].unique():

        temp = data[data["scheme_name"] == scheme].copy()

        temp = temp.sort_values("date")

        start_rows = temp[temp["date"] >= start_date]

        if len(start_rows) == 0:

            continue

        start_nav = start_rows.iloc[0]["nav"]

        end_nav = temp.iloc[-1]["nav"]

        cagr = ((end_nav/start_nav)**(1/years)-1)*100

        result.append([

            scheme,

            round(cagr,2)

        ])

    return pd.DataFrame(

        result,

        columns=[

            "scheme_name",

            f"CAGR_{years}Y"

        ]

    )

In [6]:
cagr1 = calculate_cagr(df,1)

cagr3 = calculate_cagr(df,3)

cagr5 = calculate_cagr(df,5)

In [7]:
comparison = (

    cagr1

    .merge(

        cagr3,

        on="scheme_name",

        how="outer"

    )

    .merge(

        cagr5,

        on="scheme_name",

        how="outer"

    )

)

comparison = comparison.sort_values(

    "CAGR_5Y",

    ascending=False

)

comparison.reset_index(

    drop=True,

    inplace=True

)

display(comparison)

,scheme_name,CAGR_1Y,CAGR_3Y,CAGR_5Y
0,ICICI Pru Midcap Fund - Regular - Growth,29.60,31.78,28.38
1,SBI Small Cap Fund - Regular Plan - Growth,82.78,26.67,28.03
2,DSP Small Cap Fund - Regular - Growth,65.14,27.00,27.92
3,Mirae Asset Tax Saver Fund - Regular - Growth,39.75,29.18,27.63
4,Mirae Asset Large Cap Fund - Regular - Growth,20.36,34.00,26.80
5,Kotak Flexicap Fund - Regular - Growth,26.66,29.58,26.74
6,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,53.23,32.44,26.07
7,DSP Midcap Fund - Regular - Growth,21.48,26.87,25.61
8,Axis Midcap Fund - Regular - Growth,22.26,35.11,24.45
9,SBI Bluechip Fund - Regular Plan - Growth,60.44,30.46,22.38


In [8]:
print("="*60)

print("CAGR Summary")

print("="*60)

display(

    comparison.describe()

)

CAGR Summary


,CAGR_1Y,CAGR_3Y,CAGR_5Y
count,40.000000,40.000000,40.000000
mean,19.428250,16.415250,14.541000
std,22.912863,12.206893,8.901683
min,-42.800000,-11.710000,1.030000
25%,7.382500,6.605000,6.012500
50%,17.475000,18.230000,14.475000
75%,27.165000,26.902500,21.255000
max,82.780000,35.110000,28.380000


In [10]:
import os

os.makedirs("data/processed", exist_ok=True)

comparison.to_csv(

    "C:\\Users\\tekam\\Downloads\\Data Science\\Data_Analyst_Portfolio_Project\\data\\processed\\fund_cagr_comparison.csv",

    index=False

)

print("Comparison Table Saved Successfully")

Comparison Table Saved Successfully


## Insight: CAGR Comparison

The CAGR comparison shows the long-term annualized growth performance of all mutual funds over 1-year, 3-year, and 5-year periods. Funds with consistently higher CAGR across multiple periods indicate stronger long-term performance and may be more attractive for long-term investors.

**Chart/Table Reference:** CAGR Comparison Table